In [1]:
import numpy as np
import pandas as pd
import datetime as dt
import ast

In [2]:
#pl1_file = pd.read_csv('/Users/mizwally/Desktop/DYAD_01-tangrams/DYAD_01_SCAN_011_abstract-images.csv')
pl1_file = pd.read_csv('C:\\Users\\mizwa\\Downloads\\DYAD_01_SCAN_011_abstract-images.csv')
pl2_file = pd.read_csv('C:\\Users\\mizwa\\Downloads\\DYAD_01_SCAN_012_abstract-images.csv')
pl1_ID = 'SCAN_011'
pl2_ID = 'SCAN_012'
dyad_ID = 'DYAD_01'

In [5]:
pl1_file

,Block,Round,Control?,Folder,Role,Round_start_time,Round_end_time,round_duration,completion_status,image_1,...,box_2_input,box_2_rt,box_3_input,box_3_rt,box_4_input,box_4_rt,box_5_input,box_5_rt,box_6_input,box_6_rt
0,1,1,False,easy1,director,17:31:28,17:33:29,120.648,completed,22.png,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,False,easy1,guessor,17:33:44,17:35:44,120.008,completed,22.png,...,['6'],['17:35:28'],['1'],['17:33:50'],['5'],['17:35:10'],['3'],['17:34:30'],['2'],['17:34:10']
2,2,1,False,hard1,guessor,17:35:59,17:37:59,120.012,completed,31.png,...,['3'],['17:36:51'],['5'],['17:37:34'],['2'],['17:36:31'],['4'],['17:37:18'],['1'],['17:36:07']
3,2,2,False,hard1,director,17:38:14,17:40:15,120.696,completed,23.png,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1,True,control1,director,17:40:30,17:42:31,120.552,completed,15.png,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3,2,True,control1,guessor,17:42:46,17:44:46,120.008,completed,18.png,...,['2'],['17:43:13'],['5'],['17:44:20'],['4'],['17:43:59'],['3'],['17:43:35'],['6'],['17:44:36']
6,4,1,False,easy2,guessor,17:45:01,17:47:01,120.013,completed,54.png,...,['6'],['17:46:51'],['1'],['17:46:28'],"['1', '5']","['17:45:50', '17:46:36']","['1', '2', '3']","['17:45:13', '17:45:53', '17:46:11']",['2'],['17:45:31']
7,4,2,False,easy2,director,17:47:17,17:49:17,120.680,completed,21.png,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,5,1,True,control2,director,17:49:32,17:51:33,120.660,completed,50.png,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5,2,True,control2,guessor,17:51:48,17:53:48,120.004,completed,37.png,...,['3'],['17:52:47'],['2'],['17:52:22'],['6'],['17:53:38'],"['4', '5']","['17:53:00', '17:53:17']",['4'],['17:52:53']


In [3]:
pl1_director = pl1_file[pl1_file['Role'] == 'director']
pl1_guessor = pl1_file[pl1_file['Role'] == 'guessor']
pl2_director = pl2_file[pl2_file['Role'] == 'director']
pl2_guessor = pl2_file[pl2_file['Role'] == 'guessor']

In [110]:
pl2_director.to_csv('C:\\Users\\mizwa\\Downloads\\DYAD_01_SCAN_012_director.csv', index=False)

In [106]:
pl1_rounds = []
pl2_rounds = []
for i in range(len(pl2_director.index)):
    round = {'director': list(pl2_director.iloc[i][9:15]), 
             'guessor': list(pl1_guessor.iloc[i][9:15]), 
             'inputs': list(pl1_guessor.iloc[i][15:27:2]),
             'rts': list(pl1_guessor.iloc[i][16:28:2]),
             'start_time': pl2_director.iloc[i]['Round_start_time'],
             'end_time': pl2_director.iloc[i]['Round_end_time'],
             'control': pl1_guessor.iloc[i]['Control?']}
    pl1_rounds.append(round)
for i in range(len(pl1_director.index)):
    round = {'director': list(pl1_director.iloc[i][9:15]), 
             'guessor': list(pl2_guessor.iloc[i][9:15]), 
             'inputs': list(pl2_guessor.iloc[i][15:27:2]),
             'rts': list(pl2_guessor.iloc[i][16:28:2]),
             'start_time': pl1_director.iloc[i]['Round_start_time'],
             'end_time': pl1_director.iloc[i]['Round_end_time']}
    pl2_rounds.append(round)

In [5]:
def check_answers(director, guessor, inputs, rts, start_time, diff) :
    #see when guessor image occurs in director list, then compare to answer in box
    correct = []
    control = True
    box_results = {}
    start = dt.datetime.strptime(start_time, '%H:%M:%S')
    #get accuracy, put image # and correct in slot for what box it was in
    
    for i, d in enumerate(director) :
        for g, inp, rt in zip(guessor, inputs, rts) :
            if d == g :
                control = False
                box_num = guessor.index(d)
                #calculate RT here
                #do time difference between start + 20*i-1 and rt
                #write separate rule given image display time after DYAD_09
                #calculate based on response # not image # 
                if inp == '[]' :
                    correct.append([d, None])
                    box_results[box_num] = [d, None]
                else : 
                    rt_last = ast.literal_eval(rt)[-1]
                    rt = dt.datetime.strptime(rt_last, '%H:%M:%S')
                    rt_act = rt - (start + dt.timedelta(0,(diff.total_seconds()/6)*i))
                    print('rt_dt: ', rt, 'start: ', start, 'added_time: ', (diff.total_seconds()/6)*i, 'rt: ', rt_act)
                    if i+1 == int(inp[2:1:-1]) :
                        correct.append([d, True])
                        if rt_act.total_seconds() < 0:
                            box_results[box_num] = [d, True, -1]
                        else :
                            correct.append([d, True])
                            box_results[box_num] = [d, True, rt_act.total_seconds()]
                    else :
                        if rt_act.total_seconds() < 0:
                            box_results[box_num] = [d, False, -1]
                        else :
                            correct.append([d, False])
                            box_results[box_num] = [d, False, rt_act.total_seconds()]
    print(box_results)
    if control :
        return ['control']
    else :
        #return correct
        return box_results

In [ ]:
#need to think about coding reaction time when incorrect as well, ER wants to have all answers and rts
def make_answer_key(director, guessor) :
    answer_key = {}
    control = True
    for g in guessor :
        for j, d in enumerate(director) :
            if g == d :
                control = False
                answer_key[g] = j
    if control :
        return ['control']
    else :
        return answer_key          

In [125]:
def calc_rt(start, rt, diff, i) :    
    rt = dt.datetime.strptime(rt, '%H:%M:%S')
    rt_act = rt - (start + dt.timedelta(0,(diff.total_seconds()/6)*i))
    print('rt_dt: ', rt, 'start: ', start, 'added_time: ', (diff.total_seconds()/6)*i, 'rt: ', rt_act)
    return rt_act.total_seconds()

In [ ]:
def check_correct(director, guessor, inputs, rts, start, diff) :
    #see when guessor image occurs in director list, then compare to answer in box
    graded_set = []
    #get accuracy, put image # and correct in slot for what box it was in
    
    for i, d in enumerate(director) :
        accurate = False
        correct_indices = []
        incorrect_indices = []
        rt_correct = []
        rt_incorrect = []
        for j, g, in enumerate(guessor) :
            inp = ast.literal_eval(inputs[j])
            if str(i+1) in inp :
                if d == g :      
                    correct_indices = [k for k, x in enumerate(inp) if x == str(i+1)]                    
                    incorrect_indices = [k for k, x in enumerate(inp) if x != str(i+1)]  
                    print('correct', inp, i+1, correct_indices)  
                    print('incorrect', inp, i+1, incorrect_indices)                 
                    crt_set = [ast.literal_eval(rts[j])[k] for k in correct_indices]
                    for rt in crt_set :
                        rt_correct.append(calc_rt(start, rt, diff, i))                       
                    irt_set = [ast.literal_eval(rts[j])[k] for k in incorrect_indices]
                    for rt in irt_set :
                        rt_incorrect.append(calc_rt(start, rt, diff, i))
                    if correct_indices[-1] == len(inp)-1 :
                        accurate = True 
                else :  
                    incorrect_indices = [k for k, x in enumerate(inp) if x == str(i+1)]   
                    print('incorrect', inp, i+1, incorrect_indices)
                    rt_set = [ast.literal_eval(rts[j])[k] for k in incorrect_indices]
                    for rt in rt_set :
                        rt_incorrect.append(calc_rt(start, rt, diff, i))                        
            graded = {'image': d, 'accurate?': accurate, 'correct rt': rt_correct, 'incorrect': rt_incorrect}
            graded_set.append(graded)        
        return graded_set       

In [128]:
pl1_answers = []
pl2_answers = []
for i in range(len(pl1_rounds)) :
    pl1_st = dt.datetime.strptime(pl1_rounds[i]['start_time'], '%H:%M:%S')
    pl1_end = dt.datetime.strptime(pl1_rounds[i]['end_time'], '%H:%M:%S')
    pl1_diff = pl1_end - pl1_st
    pl2_st = dt.datetime.strptime(pl2_rounds[i]['start_time'], '%H:%M:%S')
    pl2_end = dt.datetime.strptime(pl2_rounds[i]['end_time'], '%H:%M:%S')
    pl2_diff = pl2_end - pl2_st
    print(i, pl1_st, pl1_end, pl1_diff, pl1_rounds[i]['control'])
    if pl1_rounds[i]['control'] == False :
        pl1_ans = check_correct(pl1_rounds[i]['director'], pl1_rounds[i]['guessor'], pl1_rounds[i]['inputs'], 
                                pl1_rounds[i]['rts'], pl1_st, pl1_diff)
        pl1_answers.append(pl1_ans)
    else :
        pl1_answers.append(['control'])
    #pl2_ans= check_answers(pl2_rounds[i]['director'], pl2_rounds[i]['guessor'], pl2_rounds[i]['inputs'], 
                           #pl2_rounds[i]['rts'], pl2_rounds[i]['start_time'], pl2_diff)
    if pl1_ans == ['control'] :
        pl1_answers.append(pl1_ans)
    #else :
        #pl1_answers.append(dict(sorted(pl1_ans.items())))
    if pl2_ans == ['control'] :
        pl2_answers.append(pl2_ans)
    else :
        pl2_answers.append(dict(sorted(pl2_ans.items())))
#pair accuracy with reaction time and add condition to answer file

0 1900-01-01 17:33:44 1900-01-01 17:35:45 0:02:01 False
correct ['1'] 1 [0]
incorrect ['1'] 1 []
rt_dt:  1900-01-01 17:33:50 start:  1900-01-01 17:33:44 added_time:  0.0 rt:  0:00:06
1 1900-01-01 17:36:00 1900-01-01 17:38:01 0:02:01 False
correct ['1'] 1 [0]
incorrect ['1'] 1 []
rt_dt:  1900-01-01 17:36:07 start:  1900-01-01 17:36:00 added_time:  0.0 rt:  0:00:07
2 1900-01-01 17:42:47 1900-01-01 17:44:48 0:02:01 True
3 1900-01-01 17:45:03 1900-01-01 17:47:04 0:02:01 False
correct ['1'] 1 [0]
incorrect ['1'] 1 []
rt_dt:  1900-01-01 17:46:28 start:  1900-01-01 17:45:03 added_time:  0.0 rt:  0:01:25
incorrect ['1', '5'] 1 [0]
rt_dt:  1900-01-01 17:45:50 start:  1900-01-01 17:45:03 added_time:  0.0 rt:  0:00:47
incorrect ['1', '2', '3'] 1 [0]
rt_dt:  1900-01-01 17:45:13 start:  1900-01-01 17:45:03 added_time:  0.0 rt:  0:00:10
4 1900-01-01 17:51:50 1900-01-01 17:53:51 0:02:01 True
5 1900-01-01 17:56:22 1900-01-01 17:58:23 0:02:01 False
correct ['1'] 1 [0]
incorrect ['1'] 1 []
rt_dt:  1900-

In [ ]:
pl1_answer_file = pd.concat([pl1_guessor[pl1_guessor.columns[]].reset_index(drop=True), pd.Series(pl1_answers, name='Correct')], axis=1).drop(columns=['Role', 'round_duration', 'completion_status'])
#pl1_answer_file.drop(columns=['Folder', 'Role', 'Round_start_time', 'Round_end_time', 'round_duration', 'completion_stsatus'])

In [129]:
pl1_answers

[[{'image': '48.png',
   'accurate?': False,
   'correct rt': [6.0],
   'incorrect': []},
  {'image': '48.png',
   'accurate?': False,
   'correct rt': [6.0],
   'incorrect': []},
  {'image': '48.png', 'accurate?': True, 'correct rt': [6.0], 'incorrect': []},
  {'image': '48.png', 'accurate?': True, 'correct rt': [6.0], 'incorrect': []},
  {'image': '48.png', 'accurate?': True, 'correct rt': [6.0], 'incorrect': []},
  {'image': '48.png',
   'accurate?': True,
   'correct rt': [6.0],
   'incorrect': []}],
 [{'image': '30.png',
   'accurate?': False,
   'correct rt': [7.0],
   'incorrect': []},
  {'image': '30.png',
   'accurate?': False,
   'correct rt': [7.0],
   'incorrect': []},
  {'image': '30.png',
   'accurate?': False,
   'correct rt': [7.0],
   'incorrect': []},
  {'image': '30.png',
   'accurate?': False,
   'correct rt': [7.0],
   'incorrect': []},
  {'image': '30.png',
   'accurate?': False,
   'correct rt': [7.0],
   'incorrect': []},
  {'image': '30.png',
   'accurate?': Tr

In [14]:
pl1_answer_file.to_csv('/Users/mizwally/Desktop/DYAD_01-tangrams/DYAD_01_SCAN_011_answers.csv', index=False)